In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
# ============================================================
# Import Libraries
# ============================================================

from pathlib import Path
import string

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS,
    TfidfVectorizer
)

from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# ============================================================
# Dataset Paths
# ============================================================

PROJECT_DIR = Path.cwd()

KAGGLE_DATA_DIR = Path(
    "/kaggle/input/competitions/smart-mcq-solver-challenge"
)

if KAGGLE_DATA_DIR.exists():
    DATA_DIR = KAGGLE_DATA_DIR
    OUTPUT_DIR = Path("/kaggle/working")
else:
    DATA_DIR = PROJECT_DIR / "data"
    OUTPUT_DIR = PROJECT_DIR / "outputs"

OPTION_LABELS = np.array(list("ABCDE"))

In [5]:
# ============================================================
# Load Dataset
# ============================================================

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

print(train.shape)
print(test.shape)

train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
# ============================================================
# Helper Functions
# ============================================================

def clean_prompt(text):
    return text.lower().translate(
        str.maketrans("", "", string.punctuation)
    )


def combined_text(df):
    columns = ["prompt", *OPTION_LABELS]
    return df[columns].fillna("").agg(" ".join, axis=1)


def option_similarities(vectorizer, df):

    similarities = []

    for _, row in df.iterrows():

        prompt_vec = vectorizer.transform([row["prompt"]])

        scores = [
            cosine_similarity(
                prompt_vec,
                vectorizer.transform([row[option]])
            )[0, 0]

            for option in OPTION_LABELS
        ]

        similarities.append(scores)

    return np.asarray(similarities)


def rank_options(similarities):

    return [
        [
            label
            for label, _
            in sorted(
                zip(OPTION_LABELS, row),
                key=lambda x: x[1],
                reverse=True
            )
        ]

        for row in similarities
    ]


def map_at_3(y_true, preds):

    scores = []

    for truth, pred in zip(y_true, preds):

        if truth in pred:
            scores.append(1 / (pred.index(truth) + 1))
        else:
            scores.append(0)

    return np.mean(scores)

In [7]:
# ============================================================
# Exploratory Data Analysis
# ============================================================

answer_counts = train["answer"].value_counts().sort_index()

display(answer_counts)

print("Most + Least Frequent:",
      answer_counts.max() + answer_counts.min())

answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

Most + Least Frequent: 814


In [8]:
# ============================================================
# Text Cleaning
# ============================================================

cleaned_prompts = train["prompt"].map(clean_prompt)

vocab = set(
    " ".join(cleaned_prompts).split()
)

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 859


In [9]:
# ============================================================
# Stopword Removal Example
# ============================================================

row_prompt = cleaned_prompts.loc[
    train["id"] == 1
].iloc[0]

filtered_words = [

    word

    for word in row_prompt.split()

    if word not in ENGLISH_STOP_WORDS

]

print(filtered_words)
print(len(filtered_words))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
13


In [10]:
# ============================================================
# TF-IDF Baseline
# ============================================================

vectorizer = TfidfVectorizer(
    stop_words="english"
)

vectorizer.fit(
    combined_text(train)
)

print(
    "Vocabulary Size:",
    len(vectorizer.get_feature_names_out())
)

Vocabulary Size: 2762


In [11]:
# ============================================================
# Cosine Similarity
# ============================================================

train_similarity = option_similarities(
    vectorizer,
    train
)

train_rankings = rank_options(
    train_similarity
)

top1 = np.array([
    row[0]
    for row in train_rankings
])

print(
    "Row 1 Prompt vs Option A:",
    train_similarity[0,0]
)

print(
    "Top-1 Accuracy:",
    (top1 == train.answer).mean()
)

print(
    "MAP@3:",
    map_at_3(train.answer, train_rankings)
)

Row 1 Prompt vs Option A: 0.27202429519891635
Top-1 Accuracy: 0.1355
MAP@3: 0.4020916666666667


In [12]:
# ============================================================
# Majority Baseline
# ============================================================

majority = train.answer.value_counts().index[:3].tolist()

majority_preds = [
    majority
] * len(train)

print(majority)

print(
    map_at_3(
        train.answer,
        majority_preds
    )
)

['B', 'C', 'A']
0.42125


In [13]:
# ============================================================
# Generate Submission
# ============================================================

test_similarity = option_similarities(
    vectorizer,
    test
)

test_rankings = rank_options(
    test_similarity
)

submission = pd.DataFrame({

    "ID": test.id,

    "Prediction": [
        " ".join(row[:3])
        for row in test_rankings
    ]

})

OUTPUT_DIR.mkdir(exist_ok=True)

submission.to_csv(
    OUTPUT_DIR / "submission.csv",
    index=False
)

submission.head()

,ID,Prediction
0,1,A B C
1,2,A B C
2,3,A D C
3,4,A E C
4,5,A C B
